## Hylleraas molecule class

In [2]:
from hytools import visualize as hyvis
import hyobj as hyo

/home/vlew/miniconda3/envs/plotting/lib/python3.12/site-packages/numpy/core/getlimits.py:542: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)


### Distances matrix

In [3]:
import numpy as np

def get_distances(array):
    if array.ndim != 2:
        raise ValueError("Input array must be 2D")

    diff = array[:, np.newaxis, :] - array[np.newaxis, :, :]
    sq_diff = np.sum(diff ** 2, axis=-1)
    dist_matrix = np.sqrt(sq_diff)

    return dist_matrix

In [4]:
a1 = np.array([[0, 0],
               [1, 0],
               [0, 1]])
b = np.array([[-5., 3.], 
              [-2, 3] ,
              [-2, -1], 
              [1, -1], 
              [1, -5]])

dist_matrix_a1 = get_distances(a1)
dist_matrix_b = get_distances(b)

print("Array a1:\n", a1)
print("Pairwise Euclidean Distance Matrix a1:\n", dist_matrix_a1)

print("Array b:\n", b)
print("Pairwise Euclidean Distance Matrix b:\n", dist_matrix_b)

Array a1:
 [[0 0]
 [1 0]
 [0 1]]
Pairwise Euclidean Distance Matrix a1:
 [[0.         1.         1.        ]
 [1.         0.         1.41421356]
 [1.         1.41421356 0.        ]]
Array b:
 [[-5.  3.]
 [-2.  3.]
 [-2. -1.]
 [ 1. -1.]
 [ 1. -5.]]
Pairwise Euclidean Distance Matrix b:
 [[ 0.          3.          5.          7.21110255 10.        ]
 [ 3.          0.          4.          5.          8.54400375]
 [ 5.          4.          0.          3.          5.        ]
 [ 7.21110255  5.          3.          0.          4.        ]
 [10.          8.54400375  5.          4.          0.        ]]


### Methane zmat example

In [5]:
methane_default = hyo.Molecule("methane.zmat")
methane_default.coordinates

array([[ 0.      ,  0.      ,  0.      ],
       [ 1.089   ,  0.      ,  0.      ],
       [-0.362996,  1.02672 ,  0.      ],
       [-0.362996, -0.51336 , -0.889166],
       [-0.362996, -0.51336 ,  0.889166]])

In [6]:
get_distances(methane_default.coordinates)

array([[0.        , 1.089     , 1.08899957, 1.08899989, 1.08899989],
       [1.089     , 0.        , 1.77832684, 1.77832704, 1.77832704],
       [1.08899957, 1.77832684, 0.        , 1.7783314 , 1.7783314 ],
       [1.08899989, 1.77832704, 1.7783314 , 0.        , 1.778332  ],
       [1.08899989, 1.77832704, 1.7783314 , 1.778332  , 0.        ]])

In [7]:
methane_default.get_distances(units="bohr")

[[0, 1, 1.089],
 [0, 2, 1.08899956584748],
 [0, 3, 1.0889998903452653],
 [0, 4, 1.0889998903452653],
 [1, 2, 1.7783268379057882],
 [1, 3, 1.77832703661953],
 [1, 4, 1.77832703661953],
 [2, 3, 1.7783314038603717],
 [2, 4, 1.7783314038603717],
 [3, 4, 1.778332]]

## ZMAT cfour

In [8]:
def get_zmat_from_ZMAT(cfourZMATfile: str):
    """
    
    :param cfourZMATfile: 
    :return: 
    """
    with open(cfourZMATfile, "r") as f:
        cfourZMAT = f.read()
    
    blocks = cfourZMAT.split('\n\n')
    structure = blocks[0].strip().split('\n')[1:]
    values = blocks[1].strip().split('\n')
    
    value_dict = {}
    for value in values:
        key, val = value.split('=')
        key = key.strip()
        val = float(val.strip())
        value_dict[key] = val
        
    def replace_variables(line, value_dict):
        tokens = line.split()
        for i, token in enumerate(tokens):
            if token in value_dict:
                tokens[i] = str(value_dict[token])
        return ' '.join(tokens)
        
    transformed_structure = [replace_variables(line, value_dict) for line in structure]
    result = '\n'.join(transformed_structure)
    
    return result

def get_zmatLikeList(cfourZMATfile: str):
    with open(cfourZMATfile, "r") as f:
        cfourZMAT = f.read()
    blocks = cfourZMAT.split('\n\n')
    cartesianFormat = ('COORD=CARTESIAN' in cfourZMAT 
                       and 'COORD=INTERNAL' not in cfourZMAT and len(blocks)==2)
    
    if cartesianFormat:
        xyz_lines = blocks[0].strip().split('\n')[1:]
        # print(xyz_lines)
        xyz_lst = []
        for line in xyz_lines:
            line_list = line.strip().split()
            line_upd = [float(i) if line_list.index(i)!=0 else i for i in line_list]
            xyz_lst.append(line_upd)
        return xyz_lst
    else:
        structure = blocks[0].strip().split('\n')
        values = blocks[1].strip().split('\n')
    
        value_dict = {}
        for value in values:
            key, val = value.split('=')
            key = key.strip()
            val = float(val.strip())
            value_dict[key] = val
        # print(value_dict)
        def replace_variables(line, value_dict):
            tokens = line.split()
            result = []
            for token in tokens:
                if token.strip('*') in value_dict:
                    result.append(value_dict[token.strip('*')])
                else:
                    try:
                        result.append(float(token))
                    except ValueError:
                        result.append(token)
            return result
        
        transformed_structure = [replace_variables(line, value_dict) for line in structure]
        return transformed_structure[1:]

### Formamide displacement ZMAT example

In [9]:
formamide_displ_lst = get_zmatLikeList("displacementZMAT.zmat")
formamide_displ_default = hyo.Molecule(formamide_displ_lst, units='bohr')
formamide_displ_default.coordinates

array([[-2.13712924e+00, -3.91679475e-01,  6.05501616e-04],
       [ 2.15625781e+00, -3.09138858e-01, -5.73584050e-03],
       [-1.66470151e-01,  7.65935796e-01,  7.69634492e-04],
       [-8.22467789e-02,  2.84099922e+00, -5.46769240e-04],
       [ 2.30083320e+00, -2.19783621e+00,  2.45556774e-02],
       [ 3.72153584e+00,  7.48491751e-01,  3.69132166e-02]])

### Formamide ZMAT example

In [10]:
formamide_lst = get_zmatLikeList("formamideZMAT.zmat")
formamide_default = hyo.Molecule(formamide_lst)
formamide_default.coordinates

array([[ 0.      ,  0.      ,  0.      ],
       [ 2.272382,  0.      ,  0.      ],
       [ 1.05441 ,  0.592426,  0.      ],
       [ 1.120077,  1.689442, -0.      ],
       [ 2.329663, -1.000741,  0.      ],
       [ 3.111296,  0.543649,  0.      ]])

In [52]:
# view = formamide_default.view()
# view.addPropertyLabels({'property': 'x', 'showBackground': True})
# view.addPropertyLabels({'property': 'y', 'showBackground': True})
# view.addPropertyLabels({'property': 'z', 'showBackground': True})
# view.addSphere({
#     'center': {'x': 0, 'y': 0, 'z': 0},
#     'radius': 0.1,
#     'color': 'red',
#     'opacity': 1.0
# })
# view

### Formaldehyde ZMAT example

In [12]:
formaldehyde_lst = get_zmatLikeList("formaldehydeZMAT.zmat")
formaldehyde_default = hyo.Molecule(formaldehyde_lst)
formamide_default.coordinates

# view = formaldehyde_default.view()

array([[ 0.      ,  0.      ,  0.      ],
       [ 2.272382,  0.      ,  0.      ],
       [ 1.05441 ,  0.592426,  0.      ],
       [ 1.120077,  1.689442, -0.      ],
       [ 2.329663, -1.000741,  0.      ],
       [ 3.111296,  0.543649,  0.      ]])

## QUADRATURE molecules
Checking if correct displacements are used (along dimensionless normal coordinates from QUADRATURE file).
1. Make hyobj Molecules for the parsed displaced geometries from ZMAT files.
2. Make hyobj Molecules for the displacements from QUADRATURE file
3. Compare results for each normal mode.

### 1. Make hyobj Molecules for the parsed displaced geometries from ZMAT files.

In [53]:
def get_molecules_PolarDir(polar_directory: str):
    import os
    subdirs = [d for d in os.listdir(polar_directory) if os.path.isdir(d)]
    
    collected_polar_displGeos = {}
    for subdir in subdirs:
        zmatName = '/'.join([polar_directory, subdir, 'ZMAT'])
        collected_polar_displGeos[subdir] = hyo.Molecule(get_zmatLikeList(zmatName), units='bohr')
    
    return collected_polar_displGeos

### 2. Make hyobj Molecules for the displacements from QUADRATURE file

In [45]:
from calculations import parseCFOUR_forWilson
equilibrium_geometry_Molden, atomsStrs_Molden, normal_modes_Molden = parseCFOUR_forWilson.pMOLDEN('../tests/test_files_cfour/methMOLDEN')
equilibrium_geometry, freqs, normal_modes = parseCFOUR_forWilson.pQUADRATURE('../tests/test_files_cfour/methQUADRATURE')

In [48]:
def get_molecules_Quadrature(cfourQuadratureFile: str, cfourMoldenFile: str) -> dict[str:hyo.Molecule]:
    from calculations import parseCFOUR_forWilson
    equilibrium_geometry_Molden, atomsStrs_Molden, normal_modes_Molden = parseCFOUR_forWilson.pMOLDEN(cfourMoldenFile)
    equilibrium_geometry, freqs, normal_modes = parseCFOUR_forWilson.pQUADRATURE(cfourQuadratureFile)
    
    single_mode_displaced_arrays = {}
    for displ in normal_modes:
        single_mode_displaced_arrays[f'{displ}n'] = equilibrium_geometry - 0.01*normal_modes[displ]
        single_mode_displaced_arrays[f'{displ}p'] = equilibrium_geometry + 0.01*normal_modes[displ]
    
    single_mode_displaced_xyzstr = {}
    for displ in normal_modes:
        displArr_n = equilibrium_geometry - 0.01*normal_modes[displ]
        displArr_p = equilibrium_geometry + 0.01*normal_modes[displ]
        
        single_mode_displaced_xyzstr[f'{displ}n'] = []
        for na1, atom1 in enumerate(displArr_n):
            oneline1 = [atomsStrs_Molden[na1][0]]
            oneline1.extend(atom1)
            single_mode_displaced_xyzstr[f'{displ}n'].append(oneline1)
        
        single_mode_displaced_xyzstr[f'{displ}p'] = []
        for na2, atom2 in enumerate(displArr_p):
            oneline2 = [atomsStrs_Molden[na2][0]]
            oneline2.extend(atom2)
            single_mode_displaced_xyzstr[f'{displ}p'].append(oneline2)
            
    displ_molecules = {}
    for i in single_mode_displaced_xyzstr:
        displ_molecules[i] = hyo.Molecule(single_mode_displaced_xyzstr[i])
    
    return displ_molecules

In [54]:
moldenFile = '../tests/test_files_cfour/methMOLDEN'
quadratureFile = '../tests/test_files_cfour/methQUADRATURE'
# get_molecules_Quadrature(quadratureFile, moldenFile)

### 3. Compare results for each normal mode

In [55]:
fromPolDir = get_molecules_PolarDir('polar_directory')
fromQuadrature = get_molecules_Quadrature('cfourQuadratureFile', 'cfourMoldenFile')

results = {}
for dir in fromPolDir:
    results[dir] = fromPolDir[dir] == fromQuadrature[dir]

### 4. Summary as a test

In [ ]:
def test_polar_displacement_geometries():
    polar_directory = '/cluster/projects/nn14654k/vle014/refinedc4/formicac/CCSDTcc-pVQZ/polar/'
    cfourQuadratureFile = '/cluster/projects/nn14654k/vle014/refinedc4/formicac/CCSDTcc-pVQZ/polar/QUADRATURE_f'
    cfourMoldenFile = '/cluster/projects/nn14654k/vle014/refinedc4/formicac/CCSDTcc-pVQZ/polar/MOLDEN_f'
    
    fromPolDir = get_molecules_PolarDir(polar_directory)
    fromQuadrature = get_molecules_Quadrature(cfourQuadratureFile, cfourMoldenFile)
    
    results = {}
    for dir in fromPolDir:
        results[dir] = fromPolDir[dir] == fromQuadrature[dir]
        
    assert all(results.values())